# 01 Geometry Foundations

## NPL MotionLab

This notebook documents the mathematical foundations and verification of the
generic two-dimensional angle calculation used by MotionLab.

It demonstrates:

- points and vectors,
- vector subtraction,
- Euclidean magnitude,
- dot product,
- unsigned included angles,
- analytically known geometries,
- mathematical invariances,
- numerical behavior near 0° and 180°,
- the engineering rationale for the final `atan2` formulation.

The reusable implementation lives in `src/motionlab/geometry.py`.

This notebook does **not** validate camera measurements, pose estimation,
human biomechanics, or experimental accuracy.


## 1. From points to vectors

For three points \(A\), \(B\), and \(C\), with \(B\) as the angle vertex:

\[
\vec{u}=A-B
\]

\[
\vec{v}=C-B
\]

Subtracting the vertex removes absolute position and leaves the two
directions required to define the included angle.

A common translation \(t\) cancels:

\[
(A+t)-(B+t)=A-B
\]

which explains why the ideal geometric angle is translation invariant.


In [1]:
import math
import numpy as np

from motionlab.geometry import (
    angle_between_vectors_deg,
    angle_from_points_deg,
)

A = np.array([4.0, 3.0])
B = np.array([1.0, 1.0])
C = np.array([1.0, 5.0])

u = A - B
v = C - B

print("u =", u)
print("v =", v)


u = [3. 2.]
v = [0. 4.]


## 2. Vector magnitude

For a two-dimensional vector

\[
\vec{u}=(u_x,u_y)
\]

the Euclidean magnitude is

\[
\|\vec{u}\|=\sqrt{u_x^2+u_y^2}
\]

A vector of zero magnitude has no defined direction, so an angle involving
a zero-length vector is undefined.


In [2]:
vector = np.array([3.0, 4.0])

print("vector =", vector)
print("magnitude =", np.linalg.norm(vector))


vector = [3. 4.]
magnitude = 5.0


## 3. Dot product and angle

The dot product satisfies

\[
\vec{u}\cdot\vec{v}
=
\|\vec{u}\|\|\vec{v}\|\cos(\theta)
\]

which gives the familiar relationship

\[
\theta =
\arccos\left(
\frac{\vec{u}\cdot\vec{v}}
{\|\vec{u}\|\|\vec{v}\|}
\right)
\]

This formula is mathematically correct, but later numerical experiments show
that an `atan2` formulation retains more information near the boundaries
of 0° and 180°.


## 4. Verification against analytically known angles

The first vector is fixed along the positive x-axis.

The second vector is constructed from known unit-circle geometries so the
expected included angle is known independently of the MotionLab function.


In [3]:
a = (1.0, 0.0)
b = (0.0, 0.0)

known_cases = [
    (30.0,  (math.sqrt(3.0) / 2.0, 0.5)),
    (45.0,  (math.sqrt(2.0) / 2.0, math.sqrt(2.0) / 2.0)),
    (60.0,  (0.5, math.sqrt(3.0) / 2.0)),
    (90.0,  (0.0, 1.0)),
    (120.0, (-0.5, math.sqrt(3.0) / 2.0)),
    (180.0, (-1.0, 0.0)),
]

print("expected | computed           | absolute error")
print("-" * 55)

for expected, c in known_cases:
    computed = angle_from_points_deg(a, b, c)
    error = abs(computed - expected)
    print(f"{expected:8.1f} | {computed:18.15f} | {error:.3e}")


expected | computed           | absolute error
-------------------------------------------------------
    30.0 | 30.000000000000004 | 3.553e-15
    45.0 | 45.000000000000000 | 0.000e+00
    60.0 | 59.999999999999993 | 7.105e-15
    90.0 | 90.000000000000000 | 0.000e+00
   120.0 | 120.000000000000014 | 1.421e-14
   180.0 | 180.000000000000000 | 0.000e+00


## 5. Mathematical properties

For the unsigned included angle implemented here:

- common translation must preserve the angle,
- positive uniform scaling must preserve the angle,
- swapping vector order must preserve the angle.

These are mathematical properties of the measurement model, not empirical
camera-performance claims.


In [4]:
a = np.array([1.0, 0.0])
b = np.array([0.0, 0.0])
c = np.array([0.5, math.sqrt(3.0) / 2.0])

baseline = angle_from_points_deg(a, b, c)

translation = np.array([100.0, 250.0])
translated = angle_from_points_deg(
    a + translation,
    b + translation,
    c + translation,
)

scale = 7.5
scaled = angle_from_points_deg(
    a * scale,
    b * scale,
    c * scale,
)

u = a - b
v = c - b

swapped = angle_between_vectors_deg(v, u)

print("baseline   =", baseline)
print("translated =", translated)
print("scaled     =", scaled)
print("swapped    =", swapped)


baseline   = 59.99999999999999
translated = 60.00000000000028
scaled     = 59.99999999999999
swapped    = 59.99999999999999


## 6. Numerical edge case: arccos versus atan2

Near \(0^\circ\), the cosine behaves approximately as

\[
\cos(\theta)\approx 1-\frac{\theta^2}{2}
\]

while

\[
\sin(\theta)\approx\theta
\]

For very small angles, the change in cosine can therefore disappear in
floating-point arithmetic before the angular difference itself becomes
numerically unrepresentable.

The final MotionLab implementation uses

\[
\theta =
\operatorname{atan2}
\left(
|\det(\hat{u},\hat{v})|,
\hat{u}\cdot\hat{v}
\right)
\]

for the unsigned included angle.


In [5]:
def arccos_angle_deg(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)

    cosine = np.dot(u, v) / (
        np.linalg.norm(u) * np.linalg.norm(v)
    )

    return float(
        np.degrees(
            np.arccos(
                np.clip(cosine, -1.0, 1.0)
            )
        )
    )


near_zero_u = (1.0, 0.0)
near_zero_v = (1.0, 1e-8)

print(
    "arccos formulation:",
    arccos_angle_deg(near_zero_u, near_zero_v),
)

print(
    "MotionLab atan2 formulation:",
    angle_between_vectors_deg(
        near_zero_u,
        near_zero_v,
    ),
)


arccos formulation: 0.0
MotionLab atan2 formulation: 5.729577951308232e-07


## 7. Engineering conclusion

M3 verifies the generic 2D geometry layer only.

Evidence established:

- **V1, analytically verified:** known synthetic geometries and mathematical
  properties were checked against expected results.
- **V2, unit-tested:** automated tests cover known angles, invalid inputs,
  invariances, and numerical regression cases.

The `atan2` formulation was selected because it preserves the same unsigned
included-angle definition while demonstrating better numerical behavior near
0° and 180° than the tested `arccos` implementation.

This does not demonstrate:

- smartphone-camera accuracy,
- image-coordinate validity,
- pose-estimation validity,
- biomechanical validity,
- projected knee-flexion accuracy,
- agreement with a reference measurement system.

Those questions belong to later MotionLab validation layers.
